# Ball model: candidates for both CVAT clips + speed check (about 10 min)

Uses the ball model trained by `train_ball_tiles.ipynb` (`MyDrive/Playbook/ball_model/train/checkpoint_best_total.pth`).
1. Runs it over every frame of the night clip HILAL-AHLI (the day clip HILAL-HAZM is already done).
2. Saves compact candidate files for both clips (confidence >= 0.10) that Claude can read from Drive.
3. Measures speed: full-frame search (32 tiles) vs. searching only around the ball (4 tiles), with and
   without `optimize_for_inference`.

**Runtime:** T4 GPU for everything. Without a GPU (e.g. Colab GPU limit reached) it still makes the
compact copy of the day clip and skips the rest. No secrets needed. **Runtime -> Run all**; do not edit cells.

In [ ]:
# Cell 1 — GPU, Drive, repo, clip, RF-DETR
import os, sys, json, time, subprocess, shutil
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
GPU = g is not None and g.returncode == 0
print('GPU:', g.stdout.strip() if GPU else 'none (CPU runtime: only the compact copy of the day clip is made; '
      'run again with a T4 GPU later for the night clip and the speed test)')
from google.colab import drive
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/Playbook/ball_model'
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:  # a session that already ran: always use the latest code
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core')]:
    del sys.modules[m]  # re-import our code after an update
os.chdir(PB); sys.path.insert(0, PB)
CKPT = f'{DEST}/train/checkpoint_best_total.pth'
if not os.path.exists(CKPT):
    raise SystemExit('Ball model not found at MyDrive/Playbook/ball_model/train/checkpoint_best_total.pth.\n'
                     'Using another Google account? In that account\'s Drive: Shared with me -> ball_model -> '
                     'Organize -> Add shortcut -> My Drive/Playbook, then run again.')
VID = '/content/videos'; os.makedirs(VID, exist_ok=True)
CLIPS = {'hilal_ahli_B': 'HILAL-AHLI_match_B-up8.mp4'}
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', 'final-deliverable'], cwd=PB, check=True)
for name, f in CLIPS.items():
    if not os.path.exists(f'{VID}/{f}'):
        open(f'{VID}/{f}', 'wb').write(subprocess.run(['git', 'show', f'FETCH_HEAD:{f}'], cwd=PB,
                                                      capture_output=True, check=True).stdout)
if GPU:
    !pip install -q "rfdetr==1.11.0"
print('ready')

In [ ]:
# Cell 2 — Candidates for the night clip (all frames, full-frame tile search)
import cv2, numpy as np, pandas as pd
if not GPU:
    print('No GPU: skipping the night clip.')
    CLIPS = {}
else:
    from rfdetr.detr import RFDETR
    from vision.ball_model import predict_tiles, rfdetr_predict_fn
TILE = 320
if GPU:
    model = RFDETR.from_checkpoint(CKPT, trust_checkpoint=True)
    fn = rfdetr_predict_fn(model)
for name, f in CLIPS.items():
    out = f'{DEST}/{name}_ball_candidates.csv'
    if os.path.exists(out):
        print(name, 'already done'); continue
    cap = cv2.VideoCapture(f'{VID}/{f}'); rows, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        det = predict_tiles(fn, frame, tile=TILE, conf=0.05)
        rows += [{'frame': i, 'class_id': 0, 'x1': a, 'y1': b, 'x2': c, 'y2': d, 'conf': s}
                 for (a, b, c, d), s in zip(det.xyxy.tolist(), det.confidence.tolist())]
        i += 1
    cap.release()
    pd.DataFrame(rows, columns=['frame', 'class_id', 'x1', 'y1', 'x2', 'y2', 'conf']).to_csv(out, index=False)
    print(name, i, 'frames,', len(rows), 'candidates')

In [ ]:
# Cell 3 — Compact copies (conf >= 0.10, rounded) that Claude reads from Drive
for src, dst in [(f'{DEST}/hazm_ball_candidates.csv', f'{DEST}/compact_hazm.csv'),
                 (f'{DEST}/hilal_ahli_B_ball_candidates.csv', f'{DEST}/compact_ahli.csv')]:
    if not os.path.exists(src):
        print('not available yet:', os.path.basename(src)); continue
    d = pd.read_csv(src)
    d = d[d.conf >= 0.10]
    d[['x1', 'y1', 'x2', 'y2']] = d[['x1', 'y1', 'x2', 'y2']].round(0).astype(int)
    d['conf'] = d.conf.round(2)
    d[['frame', 'x1', 'y1', 'x2', 'y2', 'conf']].to_csv(dst, index=False)
    print(dst, len(d), 'rows,', os.path.getsize(dst) // 1000, 'KB')

In [ ]:
# Cell 4 — Speed: full search (32 tiles) vs. around the ball (4 tiles), plain vs. optimized
if not GPU:
    raise SystemExit('No GPU: speed test skipped. The compact day-clip file is saved; tell Claude.')
import torch
cap = cv2.VideoCapture(f'{VID}/{CLIPS["hilal_ahli_B"]}'); ok, frame = cap.read(); cap.release()
around = [(800, 400), (1056, 400), (800, 656), (1056, 656)]  # 2x2 tiles around a point
def bench(label, origins=None, n=20):
    predict_tiles(fn, frame, tile=TILE, conf=0.1, origins=origins)  # warm-up
    torch.cuda.synchronize(); t = time.time()
    for _ in range(n):
        predict_tiles(fn, frame, tile=TILE, conf=0.1, origins=origins)
    torch.cuda.synchronize()
    ms = 1000 * (time.time() - t) / n
    print(f'{label:32s} {ms:7.1f} ms/frame'); return ms
res = {'full_32_tiles': bench('full search, 32 tiles'), 'around_4_tiles': bench('around the ball, 4 tiles', around)}
try:
    model.optimize_for_inference()
    fn = rfdetr_predict_fn(model)
    res['full_32_tiles_optimized'] = bench('full search, optimized')
    res['around_4_tiles_optimized'] = bench('around the ball, optimized', around)
except Exception as e:
    print('optimize_for_inference not available:', repr(e)[:200])
json.dump(res, open(f'{DEST}/ball_speed.json', 'w'), indent=1)
print('\nDone. Tell Claude.')